In [26]:
import numpy as np
import matplotlib.pyplot as plt
from numba import njit

'''
@njit
def jacobian(u_n, h, long_N):
    J = np.zeros((long_N,long_N))
    h = np.float64(h)
    long_N = np.int64(long_N)
    
    J[0,0] = (-4 - 4 * h**2 * u_n[0]**3) / h**2
    J[0,1] = 1 / h**2
    J[long_N-1,long_N-1] = (-4 - 4 * h**2 * u_n[long_N-1]**3) / h**2
    J[long_N-1,long_N-2] = 1 / h**2
    
    for i in range(1,long_N-1):
        J[i, i] = (-4 - 4 * h**2 * u_n[i]**3) / h**2
        J[i, i-1] = 1 / h**2
        J[i, i+1] = 1 / h**2
    
    return J
'''

#the root-finding jacobian has boundary conditions of zero

def jacobian(u_n, h, N, long_N, bc_r, bc_c, int_r):
    J = np.zeros((long_N, long_N))
    
    # Boundary rows: identity
    #J[bc_r, bc_r] = 1
    
    # Interior diagonal <-- this should be whole matrix
    J[int_r, int_r] = -4 - 4 * h**2 * u_n[int_r]**3
    
    J[int_r, int_r - 1]     = 1
    J[int_r, int_r + 1]     = 1
    J[int_r, int_r - (N+1)] = 1
    J[int_r, int_r + (N+1)] = 1
    
    #J[np.ix_(int_r, bc_c)] = 0
    
    return J / h**2

@njit
def A_matrix(long_N):
    A = -4*np.identity(long_N)
    A[0,1] = 1
    A[long_N-1,long_N] = 1
    for i in range(1,long_N):
        A[i-1,i] = 1
        A[i+1,i] = 1
    return A

'''
def f(u_n, A, h, bcs):
    fu = A@u_n / h**2 - u_n**4 - bcs
    return fu
'''

def f(u_n, h, N, long_N, bc_r, int_r):
    R = np.zeros(long_N)
    
    # Boundary residual
    R[bc_r] = u_n[bc_r] - 1
    
    # Interior residual
    R[int_r] = (u_n[int_r-1] + u_n[int_r+1] + u_n[int_r-(N+1)] + u_n[int_r+(N+1)] - 4*u_n[int_r]) / h**2 - u_n[int_r]**4
    
    return R

In [27]:
N = 16 #actually using N+1 since domain is inclusive
L = 1
h = L/N
long_N = (N+1)**2

row_indices = [int(i/(N+1)) for i in range(long_N)]
row_indices = np.array(row_indices)
col_indices = [int(i%(N+1)) for i in range(long_N)]
col_indices = np.array(col_indices)

bc_r = np.where((row_indices == 0) | (row_indices == N))[0].flatten()
bc_c = np.where((col_indices == 0) | (col_indices == N))[0].flatten()
int_r = np.where((row_indices != 0) & (row_indices != N))[0].flatten()

bcs = np.zeros((long_N,1))
bcs[bc_r] -= 1
bcs[bc_c] -= 1
bcs *= h**2

u_old = np.ones((long_N,1)).flatten()

A = A_matrix(long_N)

resid = 1
tol = 1e-7

while resid > tol:
    fUn = f(u_old, h, N, long_N, bc_r, int_r)
    J = jacobian(u_old, h, N, long_N, bc_r, bc_c, int_r)
    Jinv = np.linalg.inv(J)
    u_new = u_old - Jinv@fUn
    resid = np.max(abs(u_new-u_old))
    u_old = u_new
    print('yay')
    
print('Done.')

LinAlgError: Singular matrix